In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

np.random.seed(42)

# File paths and run identifiers

SPLIT_DIR = Path("../backend/data/splits_70_15_15")
TRAIN_PATH = SPLIT_DIR / "train.parquet"
VAL_PATH = SPLIT_DIR / "val.parquet"
TEST_PATH = SPLIT_DIR / "test.parquet"

OUT_DIR = Path("../backend/data/AdaptiveR_KF")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Tag used for naming outputs 
FILE_TAG = "AdaptiveR_KF"


# Configuration

OBS_COLS = ["temperature", "humidity", "audio_density"]
D = len(OBS_COLS)

# If present, this column is used to scale Q by dt/BASE_DT_MIN (gap-aware)
DT_COL = "dt_prev_min"
BASE_DT_MIN = 15.0
USE_DT_AWARE_Q = True

MIN_ROWS_VAL = 200
MIN_ROWS_TEST = 200

# Candidate Q scaling factors (relative to the diagonal of R)
Q_SCALES = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0, 3.0]
S_JITTER = 1e-6

# Linear Gaussian model (random walk), aligned with the baseline fairness setup
A = np.eye(D)
H = np.eye(D)

# Adaptive measurement noise R (gated by normalized NIS)
ADAPT_R = True
ALPHA_R = 0.99
R_MIN_MULT = 0.2
R_MAX_MULT = 5.0
GATE_NIS_NORM = 6.0

# Missing-data inflation for state uncertainty during gaps
MISSING_Q_MULT = 1.2
MISSING_STREAK_MAX_MULT = 2.0

# Numerical safety bounds for covariance diagonals
P_DIAG_MIN = 1e-9
P_DIAG_MAX = 1e6

# Prior covariance
P0 = np.eye(D) * 10.0


# Load and clean data

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize types, remove invalid rows, and sort by hive/time.
    Ensures:
      - published_at is UTC datetime
      - tag_number is integer-like
      - observation columns are numeric
      - dt_prev_min (if present) is numeric
    """
    df = df.copy()
    df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
    df["tag_number"] = pd.to_numeric(df["tag_number"], errors="coerce").astype("Int64")
    for c in OBS_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    if DT_COL in df.columns:
        df[DT_COL] = pd.to_numeric(df[DT_COL], errors="coerce")
    df = df.dropna(subset=["published_at", "tag_number"])
    return df.sort_values(["tag_number", "published_at"]).reset_index(drop=True)

print("Loading data splits...")
train_df = clean_df(pd.read_parquet(TRAIN_PATH))
val_df = clean_df(pd.read_parquet(VAL_PATH))
test_df = clean_df(pd.read_parquet(TEST_PATH))

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
print(
    "Hives (train/val/test):",
    train_df["tag_number"].nunique(),
    val_df["tag_number"].nunique(),
    test_df["tag_number"].nunique(),
)


# Utility functions

def usable_rows(df_h: pd.DataFrame) -> int:
    """Count rows with at least one finite observation across OBS_COLS."""
    z = df_h[OBS_COLS].to_numpy(float)
    return int(np.isfinite(z).any(axis=1).sum())

def persistence_forecast_missing_safe(z: np.ndarray) -> np.ndarray:
    """
    Baseline forecast: persistence using the last observed value per dimension.
    Produces NaN until the first observation arrives for that dimension.
    """
    T, d = z.shape
    out = np.full((T, d), np.nan, float)
    last = np.full((d,), np.nan, float)
    for t in range(T):
        out[t] = last
        m = np.isfinite(z[t])
        last[m] = z[t, m]
    return out

def rmse_per_dim(y: np.ndarray, yhat: np.ndarray) -> np.ndarray:
    """Compute RMSE per dimension, ignoring NaNs."""
    y = np.asarray(y, float)
    yhat = np.asarray(yhat, float)
    out = np.full((y.shape[1],), np.nan, float)
    for d in range(y.shape[1]):
        m = np.isfinite(y[:, d]) & np.isfinite(yhat[:, d])
        if m.sum() > 0:
            out[d] = float(np.sqrt(np.mean((y[m, d] - yhat[m, d]) ** 2)))
    return out

def init_x0(z: np.ndarray) -> np.ndarray:
    """Initialize the state with the first available observation per dimension."""
    x0 = np.zeros(D, float)
    for t in range(len(z)):
        m = np.isfinite(z[t])
        if m.any():
            x0[m] = z[t, m]
            break
    return x0

# TRAIN-only normalization for aggregated RMSE (prevents leakage)
train_std = train_df[OBS_COLS].astype(float).std(skipna=True).to_numpy()
train_std = np.where(np.isfinite(train_std) & (train_std > 1e-12), train_std, 1.0)

def agg_zrmse(rmse_raw: np.ndarray) -> float:
    """Aggregate per-dimension RMSE after normalization by TRAIN standard deviation."""
    rmse_raw = np.asarray(rmse_raw, float)
    m = np.isfinite(rmse_raw) & np.isfinite(train_std) & (train_std > 1e-12)
    if m.sum() == 0:
        return np.nan
    return float(np.mean(rmse_raw[m] / train_std[m]))

def percentile_threshold(arr, p=95.0, min_n=100):
    """Return the p-th percentile threshold if there are at least min_n finite samples."""
    a = np.asarray(arr, float)
    a = a[np.isfinite(a)]
    if a.size < min_n:
        return np.nan
    return float(np.percentile(a, p))


# TRAIN-only R estimate

def estimate_R_from_train(df: pd.DataFrame) -> np.ndarray:
    """
    Estimate measurement noise covariance R using TRAIN only.
    Uses first-difference variance per hive as a proxy and aggregates via the median.
    """
    vals = []
    for _, g in df.groupby("tag_number"):
        z = g[OBS_COLS].to_numpy(float)
        if len(z) < 5:
            continue
        vals.append(0.5 * np.nanvar(np.diff(z, axis=0), axis=0))
    if len(vals) == 0:
        return np.diag([1e-3] * D)

    R_diag = np.nanmedian(np.vstack(vals), axis=0)
    R_diag = np.where(np.isfinite(R_diag) & (R_diag > 1e-8), R_diag, 1e-3)
    R_diag = np.clip(R_diag, 1e-6, None)
    return np.diag(R_diag)

R_base = estimate_R_from_train(train_df)
R0_diag = np.diag(R_base).copy()
print("\nR diag (TRAIN-only):", R0_diag)


# dt scaling (from dt_prev_min column)

def build_dt_scale(df_h: pd.DataFrame):
    """
    Build a per-timestep scaling factor for Q based on DT_COL.
    Scaling is linear in dt relative to BASE_DT_MIN and clipped to [1, 16].
    If dt scaling is disabled or DT_COL is missing, returns None.
    """
    if (not USE_DT_AWARE_Q) or (DT_COL not in df_h.columns):
        return None
    s = df_h[DT_COL].to_numpy(float) / BASE_DT_MIN
    s = np.where(np.isfinite(s) & (s > 0), s, 1.0)
    return np.clip(s, 1.0, 16.0)


# Hive selection: VAL-only (no TEST involvement)

usable_val_hives = sorted([
    int(h) for h in val_df["tag_number"].dropna().unique()
    if usable_rows(val_df[val_df["tag_number"] == h]) >= MIN_ROWS_VAL
])

if len(usable_val_hives) == 0:
    raise ValueError("No usable VAL hives. Lower MIN_ROWS_VAL or check data splits.")

# TEST-usable hives are computed for reporting only (not used for selection/tuning)
usable_test_hives = sorted([
    int(h) for h in test_df["tag_number"].dropna().unique()
    if usable_rows(test_df[test_df["tag_number"] == h]) >= MIN_ROWS_TEST
])

print("\nUsable VAL hives (selection):", len(usable_val_hives))
print("Usable TEST hives (reporting only):", len(usable_test_hives))

eval_hives = usable_val_hives  # VAL-only selection

pd.DataFrame({"hive_id": eval_hives}).to_csv(OUT_DIR / f"kf_eval_hives_{FILE_TAG}.csv", index=False)
print("Saved:", OUT_DIR / f"kf_eval_hives_{FILE_TAG}.csv")


# Chi-square quantiles (for NIS calibration)

def _chi2_ppf_wilson_hilferty(p: float, k: int) -> float:
    """
    Approximate chi-square inverse CDF using the Wilson–Hilferty transform.
    Falls back to an approximate inverse normal CDF if scipy is unavailable.
    """
    try:
        from scipy.stats import norm
        z = float(norm.ppf(p))
    except Exception:
        # Acklam inverse normal approximation
        def inv_norm_cdf(pp):
            a = [-3.969683028665376e+01, 2.209460984245205e+02,
                 -2.759285104469687e+02, 1.383577518672690e+02,
                 -3.066479806614716e+01, 2.506628277459239e+00]
            b = [-5.447609879822406e+01, 1.615858368580409e+02,
                 -1.556989798598866e+02, 6.680131188771972e+01,
                 -1.328068155288572e+01]
            c = [-7.784894002430293e-03, -3.223964580411365e-01,
                 -2.400758277161838e+00, -2.549732539343734e+00,
                 4.374664141464968e+00, 2.938163982698783e+00]
            d = [7.784695709041462e-03, 3.224671290700398e-01,
                 2.445134137142996e+00, 3.754408661907416e+00]
            plow = 0.02425
            phigh = 1 - plow
            if pp < plow:
                q = np.sqrt(-2 * np.log(pp))
                return (((((c[0] * q + c[1]) * q + c[2]) * q + c[3]) * q + c[4]) * q + c[5]) / \
                       ((((d[0] * q + d[1]) * q + d[2]) * q + d[3]) * q + 1)
            if pp > phigh:
                q = np.sqrt(-2 * np.log(1 - pp))
                return -(((((c[0] * q + c[1]) * q + c[2]) * q + c[3]) * q + c[4]) * q + c[5]) / \
                        ((((d[0] * q + d[1]) * q + d[2]) * q + d[3]) * q + 1)
            q = pp - 0.5
            r = q * q
            return (((((a[0] * r + a[1]) * r + a[2]) * r + a[3]) * r + a[4]) * r + a[5]) * q / \
                   (((((b[0] * r + b[1]) * r + b[2]) * r + b[3]) * r + b[4]) * r + 1)

        z = float(inv_norm_cdf(p))

    k = max(int(k), 1)
    return float(k * (1 - 2 / (9 * k) + z * np.sqrt(2 / (9 * k))) ** 3)

def chi2_ppf(p: float, k: int) -> float:
    """Prefer scipy.stats.chi2.ppf; otherwise use Wilson–Hilferty approximation."""
    try:
        from scipy.stats import chi2
        return float(chi2.ppf(p, df=int(k)))
    except Exception:
        return _chi2_ppf_wilson_hilferty(p, k)


# KF (fair forecast) with gated Adaptive-R

def kf_run_fair(
    z: np.ndarray,
    Q_base: np.ndarray,
    R_init: np.ndarray,
    x0: np.ndarray,
    P0: np.ndarray,
    dt_scale=None,
    adapt_r: bool = True,
    alpha_r: float = 0.99,
    r_min_mult: float = 0.2,
    r_max_mult: float = 5.0,
    gate_nis_norm: float = 6.0,
    missing_q_mult: float = 1.2,
    missing_streak_max_mult: float = 2.0,
    P_DIAG_MIN: float = 1e-9,
    P_DIAG_MAX: float = 1e6,
):
    """
    Run a Kalman filter with "fair" forecast evaluation:
      - x_pred[t] and P_pred[t] are recorded BEFORE assimilating z[t].
      - Only observed dimensions at time t are assimilated.
      - Measurement noise R can be adapted online (gated by normalized NIS).
    """
    T = len(z)
    x = x0.copy()
    P = P0.copy()

    x_pred = np.zeros((T, D), float)
    P_pred = np.zeros((T, D, D), float)
    x_filt = np.zeros((T, D), float)

    nis = np.full(T, np.nan, float)
    nis_norm = np.full(T, np.nan, float)
    nis_dof = np.zeros(T, int)

    I = np.eye(D)

    R0 = np.diag(R_init).copy()
    R_diag_t = R0.copy()
    R_diag_hist = np.zeros((T, D), float)

    missing_streak = 0

    def clamp_P(Pmat: np.ndarray) -> np.ndarray:
        """Enforce symmetry and constrain covariance diagonals for numerical stability."""
        Pmat = 0.5 * (Pmat + Pmat.T)
        d = np.clip(np.diag(Pmat), P_DIAG_MIN, P_DIAG_MAX)
        Pmat = Pmat.copy()
        np.fill_diagonal(Pmat, d)
        if not np.isfinite(Pmat).all():
            Pmat = np.diag(d)
        return Pmat

    for t in range(T):
        scale = 1.0 if dt_scale is None else float(dt_scale[t])
        Q_t = Q_base * scale

        # Forecast step 
        x = A @ x
        P = A @ P @ A.T + Q_t
        P = clamp_P(P)

        #  Missing-observation inflation (applied before saving P_pred) 
        zt = z[t]
        mask = np.isfinite(zt)
        m_dim = int(mask.sum())
        nis_dof[t] = m_dim

        if m_dim == 0:
            missing_streak += 1
            eff = min(1.0 + (missing_q_mult - 1.0) * missing_streak, missing_streak_max_mult)
            d = np.clip(np.diag(P) * eff, P_DIAG_MIN, P_DIAG_MAX)
            P = np.diag(d)
        else:
            missing_streak = 0

        # FAIR outputs (pre-update)
        x_pred[t] = x
        P_pred[t] = P

        if m_dim == 0:
            x_filt[t] = x
            R_diag_hist[t] = R_diag_t
            continue

        Ht = H[mask]
        y = zt[mask] - (Ht @ x)

        # NIS computed using current R (before any adaptive update at this step)
        Rt0 = np.diag(R_diag_t)[np.ix_(mask, mask)]
        S0 = Ht @ P @ Ht.T + Rt0
        S0 = 0.5 * (S0 + S0.T) + S_JITTER * np.eye(m_dim)

        try:
            nis_val = float(y.T @ np.linalg.solve(S0, y))
        except np.linalg.LinAlgError:
            nis_val = float(y.T @ np.linalg.pinv(S0) @ y)

        nis[t] = nis_val
        nis_norm[t] = nis_val / max(m_dim, 1)

        # ---- Gated Adaptive-R update ----
        if adapt_r and np.isfinite(nis_norm[t]) and (nis_norm[t] <= gate_nis_norm):
            idx = np.where(mask)[0]
            R_diag_t[idx] = alpha_r * R_diag_t[idx] + (1.0 - alpha_r) * (y ** 2)
            R_diag_t = np.clip(R_diag_t, r_min_mult * R0, r_max_mult * R0)

        #  Measurement update 
        Rt = np.diag(R_diag_t)[np.ix_(mask, mask)]
        S = Ht @ P @ Ht.T + Rt
        S = 0.5 * (S + S.T) + S_JITTER * np.eye(m_dim)

        try:
            K = (P @ Ht.T) @ np.linalg.solve(S, np.eye(m_dim))
        except np.linalg.LinAlgError:
            K = (P @ Ht.T) @ np.linalg.pinv(S)

        x = x + K @ y
        P = (I - K @ Ht) @ P @ (I - K @ Ht).T + K @ Rt @ K.T
        P = clamp_P(P)

        x_filt[t] = x
        R_diag_hist[t] = R_diag_t

    return x_pred, x_filt, P_pred, nis, nis_norm, nis_dof, R_diag_hist

# Q tuning (VAL only, fixed R)

tuning = []
for s in Q_SCALES:
    Q = np.diag(R0_diag * float(s))
    scores = []
    for h in eval_hives:
        vh = val_df[val_df["tag_number"].astype(int) == h]
        z = vh[OBS_COLS].to_numpy(float)

        x_pred, *_ = kf_run_fair(
            z,
            Q_base=Q,
            R_init=R_base,
            x0=init_x0(z),
            P0=P0,
            dt_scale=build_dt_scale(vh),
            adapt_r=False,  # fixed R during Q tuning
            missing_q_mult=MISSING_Q_MULT,
            missing_streak_max_mult=MISSING_STREAK_MAX_MULT,
            P_DIAG_MIN=P_DIAG_MIN,
            P_DIAG_MAX=P_DIAG_MAX,
        )
        scores.append(agg_zrmse(rmse_per_dim(z, x_pred)))

    tuning.append({"Q_scale": float(s), "median_val_agg_zrmse": float(np.nanmedian(scores))})

tuning_df = pd.DataFrame(tuning).sort_values("median_val_agg_zrmse").reset_index(drop=True)
best_scale = float(tuning_df.iloc[0]["Q_scale"])
Q_best = np.diag(R0_diag * best_scale)

tuning_path = OUT_DIR / f"kf_q_tuning_results_{FILE_TAG}.csv"
tuning_df.to_csv(tuning_path, index=False)

print("\nQ tuning results (VAL-selected hives only):")
print(tuning_df.to_string(index=False))
print("\nBest Q_scale:", best_scale)
print("Saved tuning:", tuning_path)


# Run final model and export per-step results

rows = []
for split, df, min_rows in [("val", val_df, MIN_ROWS_VAL), ("test", test_df, MIN_ROWS_TEST)]:
    present_hives = set(df["tag_number"].dropna().astype(int).unique())
    run_hives = [h for h in eval_hives if h in present_hives]

    for h in run_hives:
        dh = df[df["tag_number"].astype(int) == h]
        if usable_rows(dh) < min_rows:
            continue

        z = dh[OBS_COLS].to_numpy(float)

        x_pred, x_filt, P_pred, nis, nis_norm, nis_dof, R_diag_hist = kf_run_fair(
            z,
            Q_base=Q_best,
            R_init=R_base,
            x0=init_x0(z),
            P0=P0,
            dt_scale=build_dt_scale(dh),
            adapt_r=ADAPT_R,
            alpha_r=ALPHA_R,
            r_min_mult=R_MIN_MULT,
            r_max_mult=R_MAX_MULT,
            gate_nis_norm=GATE_NIS_NORM,
            missing_q_mult=MISSING_Q_MULT,
            missing_streak_max_mult=MISSING_STREAK_MAX_MULT,
            P_DIAG_MIN=P_DIAG_MIN,
            P_DIAG_MAX=P_DIAG_MAX,
        )

        pred_var = np.maximum(np.diagonal(P_pred, axis1=1, axis2=2), 0.0)
        pred_std = np.sqrt(pred_var)

        tmp = pd.DataFrame({
            "published_at": pd.to_datetime(dh["published_at"].to_numpy(), utc=True),
            "hive_id": int(h),
            "split": split,

            "y_true_temperature": z[:, 0],
            "y_true_humidity": z[:, 1],
            "y_true_audio_density": z[:, 2],

            "y_pred_temperature": x_pred[:, 0],
            "y_pred_humidity": x_pred[:, 1],
            "y_pred_audio_density": x_pred[:, 2],

            "pred_std_temperature": pred_std[:, 0],
            "pred_std_humidity": pred_std[:, 1],
            "pred_std_audio_density": pred_std[:, 2],

            "kf_filt_temperature": x_filt[:, 0],
            "kf_filt_humidity": x_filt[:, 1],
            "kf_filt_audio_density": x_filt[:, 2],

            "nis_raw": nis,
            "nis_norm": nis_norm,
            "nis_dof": nis_dof,

            "R_diag_temperature": R_diag_hist[:, 0],
            "R_diag_humidity": R_diag_hist[:, 1],
            "R_diag_audio_density": R_diag_hist[:, 2],

            # Reporting-only: whether the hive meets the TEST usability criterion
            "is_test_usable_hive": (int(h) in set(usable_test_hives)),
        })
        rows.append(tmp)

out_df = pd.concat(rows, ignore_index=True)


# VAL-only thresholds (dof > 0):
#  - Percentiles for anomaly flagging (data-driven)
#  - Chi-square thresholds for consistency checks (theoretical)

val_mask = (out_df["split"] == "val") & (out_df["nis_dof"] > 0) & np.isfinite(out_df["nis_norm"])
val_scores = out_df.loc[val_mask, "nis_norm"].to_numpy(float)

thr95_pct = percentile_threshold(val_scores, 95.0, min_n=100)
thr99_pct = percentile_threshold(val_scores, 99.0, min_n=100)

print("\n=== VAL-only percentile anomaly thresholds ===")
print(f"nis_norm p95={thr95_pct:.6f}  p99={thr99_pct:.6f} | samples={int(np.isfinite(val_scores).sum())}")

out_df["is_anomaly_norm_pct_p95"] = (
    (out_df["nis_dof"] > 0) &
    np.isfinite(out_df["nis_norm"]) &
    np.isfinite(thr95_pct) &
    (out_df["nis_norm"] > thr95_pct)
) if np.isfinite(thr95_pct) else np.zeros(len(out_df), dtype=bool)

out_df["is_anomaly_norm_pct_p99"] = (
    (out_df["nis_dof"] > 0) &
    np.isfinite(out_df["nis_norm"]) &
    np.isfinite(thr99_pct) &
    (out_df["nis_norm"] > thr99_pct)
) if np.isfinite(thr99_pct) else np.zeros(len(out_df), dtype=bool)

# Classical consistency checks: nis_raw > chi2_ppf(p, dof)
out_df["chi2_thr_raw_p95"] = out_df["nis_dof"].apply(lambda k: chi2_ppf(0.95, int(k)) if int(k) > 0 else np.nan)
out_df["chi2_thr_raw_p99"] = out_df["nis_dof"].apply(lambda k: chi2_ppf(0.99, int(k)) if int(k) > 0 else np.nan)

out_df["is_inconsistent_chi2_p95"] = (
    (out_df["nis_dof"] > 0) &
    np.isfinite(out_df["nis_raw"]) &
    np.isfinite(out_df["chi2_thr_raw_p95"]) &
    (out_df["nis_raw"] > out_df["chi2_thr_raw_p95"])
)
out_df["is_inconsistent_chi2_p99"] = (
    (out_df["nis_dof"] > 0) &
    np.isfinite(out_df["nis_raw"]) &
    np.isfinite(out_df["chi2_thr_raw_p99"]) &
    (out_df["nis_raw"] > out_df["chi2_thr_raw_p99"])
)


# Save per-timestep exports

parquet_all = OUT_DIR / f"kf_val_test_forecast_nis_{FILE_TAG}.parquet"
parquet_val = OUT_DIR / f"kf_forecasts_val_{FILE_TAG}.parquet"
parquet_test = OUT_DIR / f"kf_forecasts_test_{FILE_TAG}.parquet"

out_df.to_parquet(parquet_all, index=False)
out_df[out_df["split"] == "val"].to_parquet(parquet_val, index=False)
out_df[out_df["split"] == "test"].to_parquet(parquet_test, index=False)

print("\nSaved per-timestep outputs:")
print(" -", parquet_all)
print(" -", parquet_val)
print(" -", parquet_test)


# Summary metrics (per hive)

summary_rows = []
for split_name in ["val", "test"]:
    df_step = out_df[out_df["split"] == split_name].copy()
    for hive_id, g in df_step.groupby("hive_id"):
        z = g[["y_true_temperature", "y_true_humidity", "y_true_audio_density"]].to_numpy(float)
        base = persistence_forecast_missing_safe(z)
        pred = g[["y_pred_temperature", "y_pred_humidity", "y_pred_audio_density"]].to_numpy(float)

        rmse_base = rmse_per_dim(z, base)
        rmse_kf = rmse_per_dim(z, pred)

        summary_rows.append({
            "hive_id": int(hive_id),
            "split": split_name,
            "baseline_agg_zrmse": agg_zrmse(rmse_base),
            "kf_agg_zrmse": agg_zrmse(rmse_kf),
            "n_rows": int(len(g)),
            "is_test_usable_hive": bool(g["is_test_usable_hive"].iloc[0]),
        })

summary_df = pd.DataFrame(summary_rows)
summary_path = OUT_DIR / f"kf_summary_metrics_per_hive_{FILE_TAG}.csv"
summary_df.to_csv(summary_path, index=False)


# Run configuration (for reproducibility)

run_config_path = OUT_DIR / f"kf_run_config_{FILE_TAG}.csv"
pd.DataFrame([{
    "FILE_TAG": FILE_TAG,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),

    "best_Q_scale": best_scale,
    "USE_DT_AWARE_Q": USE_DT_AWARE_Q,
    "DT_COL": DT_COL,
    "BASE_DT_MIN": BASE_DT_MIN,

    "R_diag_init": ",".join([f"{x:.6g}" for x in R0_diag]),
    "P0_diag": ",".join([f"{x:.6g}" for x in np.diag(P0)]),

    "ADAPT_R": ADAPT_R,
    "ALPHA_R": ALPHA_R,
    "R_MIN_MULT": R_MIN_MULT,
    "R_MAX_MULT": R_MAX_MULT,
    "GATE_NIS_NORM": GATE_NIS_NORM,

    "MISSING_Q_MULT": MISSING_Q_MULT,
    "MISSING_STREAK_MAX_MULT": MISSING_STREAK_MAX_MULT,
    "P_DIAG_MIN": P_DIAG_MIN,
    "P_DIAG_MAX": P_DIAG_MAX,

    # Percentile anomaly thresholds (VAL-derived)
    "nis_norm_pct_p95_val": thr95_pct,
    "nis_norm_pct_p99_val": thr99_pct,
    "val_nis_samples_used": int(np.isfinite(val_scores).sum()),

    "eval_hives_val_count": int(len(eval_hives)),
    "test_usable_hives_reporting_only": int(len(usable_test_hives)),
    "OUT_DIR": str(OUT_DIR),
}]).to_csv(run_config_path, index=False)

print("\nSaved summary:", summary_path)
print("Saved run config:", run_config_path)
print("DONE — Adaptive-R gated KF: VAL-only hive selection, percentile anomaly flags, and chi-square calibration.")

Loading data splits...
Train: (560733, 31) Val: (120160, 31) Test: (120187, 31)
Hives (train/val/test): 52 52 52

R diag (TRAIN-only): [0.0124549  0.29751825 2.82167299]

Usable VAL hives (selection): 33
Usable TEST hives (reporting only): 27
Saved: ..\backend\data\AdaptiveR_KF\kf_eval_hives_AdaptiveR_KF.csv

Q tuning results (VAL-selected hives only):
 Q_scale  median_val_agg_zrmse
  3.0000              0.168707
  1.0000              0.171058
  0.3000              0.184342
  0.1000              0.212348
  0.0300              0.265609
  0.0100              0.335481
  0.0030              0.404612
  0.0010              0.465565
  0.0003              0.520378
  0.0001              0.565352

Best Q_scale: 3.0
Saved tuning: ..\backend\data\AdaptiveR_KF\kf_q_tuning_results_AdaptiveR_KF.csv

=== VAL-only percentile anomaly thresholds ===
nis_norm p95=1.961865  p99=5.370758 | samples=57425

Saved per-timestep outputs:
 - ..\backend\data\AdaptiveR_KF\kf_val_test_forecast_nis_AdaptiveR_KF.parque